# Show3DSlices

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/electronmicroscopy/quantem.widget/blob/main/docs/tutorials/show3dslices.ipynb)

`Show3DSlices` links a top slice, an arbitrary-angle side cut, and a 3D volume view. Move the crosshair and rotate the oblique cut to inspect how features extend through depth.

```{tip}
Run this exact notebook with the Colab badge above, or [View or download this notebook on GitHub](https://github.com/electronmicroscopy/quantem.widget/blob/main/docs/tutorials/show3dslices.ipynb). For finished results, use [HTML and file export](widget_export) to export interactive HTML or share a trusted notebook with widget state.
```


In [ ]:
# @title Install QuantEM { display-mode: "form" }
if get_ipython().__class__.__module__.startswith("google.colab"):
    !wget -q https://raw.githubusercontent.com/electronmicroscopy/quantem.widget/main/scripts/install_colab.py -O install_quantem.py
    %run install_quantem.py


In [ ]:
import quantem.widget as qw
qw.profile()

import numpy as np

# Synthetic scalar inspection volume, not a reconstructed or simulated potential:
# Gaussian columns drift across
# depth, so scrubbing through z shows the lattice shift slice by slice.
depth, size = 24, 256
row, col = np.mgrid[0:size, 0:size]
spacing = 20
sites = np.arange(spacing // 2, size, spacing)
volume = np.zeros((depth, size, size), dtype=np.float32)
for z in range(depth):
    shift = 6.0 * np.sin(2 * np.pi * z / depth)  # columns sway with depth
    for r0 in sites:
        for c0 in sites:
            volume[z] += np.exp(-(((row - r0 - shift) ** 2 + (col - c0) ** 2)) / (2 * 2.5 ** 2))
volume.shape, volume.dtype

## Linked top and side cuts

Pass the stack with depth and lateral calibration so `Show3DSlices` can draw calibrated scale bars. Move the crosshair on the top slice to position the side cut. Adjust the oblique angle to inspect a different direction through the columns. The 3D view shows where those planes cut the volume.


In [ ]:
from quantem.widget import Show3DSlices

slices = Show3DSlices(
    volume,
    sampling=(0.3, 0.18, 0.18),
    units=("nm", "nm", "nm"),
    title="Synthetic column volume (arbitrary units)",
)
slices


## Use a reconstructed volume

The array order is `(depth, row, col)`. Sampling supplies the spacing along those same axes; use the actual reconstruction spacing rather than the nominal specimen thickness divided by an assumed count. These synthetic Gaussian values have arbitrary units and are not calibrated atomic potential or phase.

Change the colormap, move the linked crosshair, and rotate the side-cut angle. The depth stretch helps display anisotropic voxels; it does not change their calibration. For one depth slider, playback, and a moving average, open the same volume in **Show3D** below. A three-plane mean helps inspection but does not improve the reconstruction's physical resolution.


In [ ]:
scrubber = qw.Show3D(volume, sampling=0.18, units="nm", avg_window=3)
scrubber


## Keep the comparison interactive

```python
slices.export_html("depth-slices.html", encoding="full")
scrubber.export_html("depth-playback.html", encoding="full")
```

`encoding="full"` preserves the display values rather than quantizing them for a smaller browse file. See [HTML export](widget_export) for packaging and browser requirements. Continue with [ShowCIF](showcif.ipynb) to inspect a known atomic model, or the [demo route](demo.md) for the complete sequence.
